# 🛒 Sales Data Analysis & Forecasting
### Full Notebook: Load → Clean → EDA → Features → Forecast → Dashboard
---
**Dataset:** `retail_sales.csv` (2022–2023 Retail Data)  
**Models:** Linear Regression · Random Forest · Gradient Boosting

## ⚙️ 0 — Install & Import

In [ ]:
# !pip install pandas numpy matplotlib seaborn scikit-learn plotly
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi':110, 'axes.grid':True, 'grid.alpha':0.3})
OUTPUT_DIR = '../outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Imports OK ✅')

## 📂 1 — Load Dataset

In [ ]:
df = pd.read_csv('../data/retail_sales.csv', parse_dates=['Date'])
print(f'Shape: {df.shape}')
print(f'Date range: {df.Date.min().date()} → {df.Date.max().date()}')
df.head()

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())
df.describe()

## 🧹 2 — Data Cleaning & Preprocessing

In [ ]:
df.drop_duplicates(inplace=True)
df.dropna(subset=['Date','Revenue'], inplace=True)

df['Year']       = df['Date'].dt.year
df['Month']      = df['Date'].dt.month
df['Quarter']    = df['Date'].dt.quarter
df['MonthName']  = df['Date'].dt.strftime('%b')
df['YearMonth']  = df['Date'].dt.to_period('M')
df['Profit_Margin_%'] = (df['Profit'] / df['Revenue'] * 100).round(2)

print(f'Clean shape: {df.shape}')
df[['Date','Category','Revenue','Profit','Profit_Margin_%']].head()

## 📊 3 — Exploratory Data Analysis (EDA)

In [ ]:
# 3A — Monthly Revenue Trend
monthly = df.groupby('YearMonth')['Revenue'].sum().reset_index()
monthly['YM'] = monthly['YearMonth'].astype(str)

fig, ax = plt.subplots(figsize=(14,4))
ax.fill_between(range(len(monthly)), monthly['Revenue'], alpha=0.2, color='steelblue')
ax.plot(range(len(monthly)), monthly['Revenue'], 'o-', color='steelblue', lw=2, ms=5)
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['YM'], rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.set_title('Monthly Revenue Trend (2022–2023)', fontsize=13, fontweight='bold')
ax.set_ylabel('Revenue ($)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/monthly_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3B — Revenue & Profit by Category
cat_rev  = df.groupby('Category')['Revenue'].sum().sort_values(ascending=False)
cat_prof = df.groupby('Category')['Profit'].sum()

fig, axes = plt.subplots(1, 2, figsize=(13,4))
for ax, data, title, color in zip(
        axes,
        [cat_rev, cat_prof],
        ['Total Revenue by Category','Total Profit by Category'],
        ['steelblue','green']):
    bars = ax.bar(data.index, data.values, color=color, edgecolor='black', alpha=0.85)
    ax.bar_label(bars, fmt='${:,.0f}', fontsize=8, padding=3)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
    ax.set_title(title, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/revenue_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3C — Revenue by Region (line chart)
reg_m = df.groupby(['YearMonth','Region'])['Revenue'].sum().reset_index()
reg_m['YM'] = reg_m['YearMonth'].astype(str)

fig, ax = plt.subplots(figsize=(14,4))
for reg in df['Region'].unique():
    sub = reg_m[reg_m['Region']==reg]
    ax.plot(sub['YM'], sub['Revenue'], 'o-', lw=1.8, ms=4, label=reg)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.set_title('Monthly Revenue by Region', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/revenue_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3D — Top 10 Products
top_prod = df.groupby('Product')['Revenue'].sum().nlargest(10).sort_values()
fig, ax = plt.subplots(figsize=(10,5))
bars = ax.barh(top_prod.index, top_prod.values,
               color=sns.color_palette('coolwarm', len(top_prod)))
ax.bar_label(bars, fmt='${:,.0f}', fontsize=8, padding=3)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.set_title('Top 10 Products by Revenue', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/top_products.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3E — Quarterly Heatmap
pivot = df.pivot_table(values='Revenue', index='Category', columns='Quarter', aggfunc='sum')
pivot.columns = [f'Q{c}' for c in pivot.columns]

fig, ax = plt.subplots(figsize=(8,4))
sns.heatmap(pivot, annot=True, fmt=',.0f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Revenue Heatmap — Category × Quarter', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/quarterly_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3F — Profit Margin Distribution
fig, ax = plt.subplots(figsize=(9,4))
for cat in df['Category'].unique():
    ax.hist(df[df['Category']==cat]['Profit_Margin_%'], bins=15, alpha=0.6, label=cat)
ax.set_title('Profit Margin Distribution by Category', fontsize=13, fontweight='bold')
ax.set_xlabel('Profit Margin (%)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/profit_margin_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔧 4 — Feature Engineering

In [ ]:
monthly_fe = (df.groupby('YearMonth')['Revenue']
              .sum().reset_index().sort_values('YearMonth'))

monthly_fe['Revenue_Lag1']   = monthly_fe['Revenue'].shift(1)
monthly_fe['Revenue_Lag2']   = monthly_fe['Revenue'].shift(2)
monthly_fe['Revenue_Lag3']   = monthly_fe['Revenue'].shift(3)
monthly_fe['Rolling_3M_Avg'] = monthly_fe['Revenue'].rolling(3).mean()
monthly_fe['Rolling_3M_Std'] = monthly_fe['Revenue'].rolling(3).std()
monthly_fe['MoM_Growth_%']   = monthly_fe['Revenue'].pct_change() * 100
monthly_fe['Month_Num']      = range(1, len(monthly_fe)+1)
monthly_fe['Month']          = monthly_fe['YearMonth'].apply(lambda x: x.month)
monthly_fe['Year']           = monthly_fe['YearMonth'].apply(lambda x: x.year)
monthly_fe.dropna(inplace=True)

print(f'Monthly records for modelling: {len(monthly_fe)}')
monthly_fe[['YearMonth','Revenue','Revenue_Lag1','Rolling_3M_Avg','MoM_Growth_%']].tail()

In [ ]:
# Month-over-Month Growth Chart
fig, ax = plt.subplots(figsize=(12,4))
colors = ['green' if x >= 0 else 'red' for x in monthly_fe['MoM_Growth_%']]
ax.bar(monthly_fe['Month_Num'], monthly_fe['MoM_Growth_%'],
       color=colors, edgecolor='black', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Month-over-Month Revenue Growth (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Month Index'); ax.set_ylabel('Growth (%)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/mom_growth.png', dpi=150, bbox_inches='tight')
plt.show()

## 🤖 5 — Sales Forecasting Models

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feature_cols = ['Month_Num','Month','Year',
                'Revenue_Lag1','Revenue_Lag2','Revenue_Lag3',
                'Rolling_3M_Avg','Rolling_3M_Std']
X = monthly_fe[feature_cols]
y = monthly_fe['Revenue']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

models = {
    'Linear Regression':  LinearRegression(),
    'Random Forest':      RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    results[name] = {'MAE':mae,'RMSE':rmse,'R²':r2,'model':model,'preds':preds}
    print(f'{name:25s} MAE=${mae:,.0f}  RMSE=${rmse:,.0f}  R²={r2:.4f}')

best_model_name = max(results, key=lambda k: results[k]['R²'])
print(f'\n🏆 Best Model: {best_model_name}')

In [ ]:
# Actual vs Predicted
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(monthly_fe['Month_Num'], monthly_fe['Revenue'],
        'o-', color='steelblue', lw=2, label='Actual')
test_idx = monthly_fe['Month_Num'].values[-len(y_test):]
for name, res in results.items():
    ax.plot(test_idx, res['preds'], '--', lw=1.5, label=f'{name} (pred)')
ax.axvline(test_idx[0]-0.5, color='red', linestyle=':', label='Train/Test Split')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.set_title('Actual vs Predicted Revenue', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 6-Month Forecast
best = results[best_model_name]['model']
last = monthly_fe.iloc[-1]
lag1, lag2, lag3 = last['Revenue'], last['Revenue_Lag1'], last['Revenue_Lag2']
roll_avg, roll_std = last['Rolling_3M_Avg'], last['Rolling_3M_Std']
month_num, month, year = int(last['Month_Num'])+1, int(last['Month']), int(last['Year'])

future = []
for _ in range(6):
    month += 1
    if month > 12: month=1; year+=1
    row = {'Month_Num':month_num,'Month':month,'Year':year,
           'Revenue_Lag1':lag1,'Revenue_Lag2':lag2,'Revenue_Lag3':lag3,
           'Rolling_3M_Avg':roll_avg,'Rolling_3M_Std':roll_std}
    pred = best.predict(pd.DataFrame([row]))[0]
    future.append({'Period':f'{year}-{month:02d}','Forecast_Revenue':pred})
    lag3,lag2,lag1 = lag2,lag1,pred
    roll_avg = (roll_avg*2+pred)/3
    month_num += 1

future_df = pd.DataFrame(future)
print('6-Month Revenue Forecast:')
future_df['Forecast_Revenue'] = future_df['Forecast_Revenue'].map('${:,.2f}'.format)
print(future_df.to_string(index=False))

In [ ]:
# Forecast Chart
future_df['Forecast_Revenue'] = future_df['Forecast_Revenue'].str.replace('$','').str.replace(',','').astype(float)
hist_rev = monthly_fe['Revenue'].tolist()
last_mn  = int(monthly_fe['Month_Num'].iloc[-1])
future_rev = [hist_rev[-1]] + future_df['Forecast_Revenue'].tolist()

fig, ax = plt.subplots(figsize=(14,5))
ax.plot(monthly_fe['Month_Num'], monthly_fe['Revenue'],
        'o-', color='steelblue', lw=2, label='Historical')
ax.plot(range(last_mn, last_mn+7), future_rev,
        's--', color='orangered', lw=2, ms=8, label='6-Month Forecast')
ax.fill_between(range(last_mn, last_mn+7),
                [v*0.9 for v in future_rev],[v*1.1 for v in future_rev],
                alpha=0.15, color='orangered', label='±10% Band')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.set_title(f'Revenue Forecast — Next 6 Months ({best_model_name})',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/revenue_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 🌐 6 — Interactive Plotly Dashboard

In [ ]:
fig = make_subplots(rows=2, cols=2,
    subplot_titles=('Monthly Revenue Trend','Revenue by Category',
                    'Revenue by Region','Top Products'),
    vertical_spacing=0.14, horizontal_spacing=0.10)

mp = df.groupby('YearMonth')['Revenue'].sum().reset_index()
mp['YM'] = mp['YearMonth'].astype(str)
fig.add_trace(go.Scatter(x=mp['YM'], y=mp['Revenue'],
    mode='lines+markers', name='Revenue', line=dict(color='steelblue',width=2),
    fill='tozeroy',fillcolor='rgba(70,130,180,0.1)'), row=1,col=1)

cat = df.groupby('Category')['Revenue'].sum().sort_values(ascending=False)
fig.add_trace(go.Bar(x=cat.index, y=cat.values, name='Category',
    marker_color=px.colors.qualitative.Bold[:len(cat)],showlegend=False), row=1,col=2)

rm = df.groupby(['YearMonth','Region'])['Revenue'].sum().reset_index()
rm['YM'] = rm['YearMonth'].astype(str)
for reg in df['Region'].unique():
    s = rm[rm['Region']==reg]
    fig.add_trace(go.Scatter(x=s['YM'],y=s['Revenue'],
        mode='lines+markers',name=reg,line=dict(width=1.8)), row=2,col=1)

tp = df.groupby('Product')['Revenue'].sum().nlargest(10).sort_values()
fig.add_trace(go.Bar(x=tp.values, y=tp.index, orientation='h',
    marker_color=px.colors.sequential.RdBu[:len(tp)],
    name='Products', showlegend=False), row=2,col=2)

fig.update_layout(title='🛒 Sales Analysis Dashboard',
    title_font_size=20, height=750, template='plotly_white')
fig.update_yaxes(tickprefix='$', tickformat=',.0f', row=1, col=1)
fig.update_yaxes(tickprefix='$', tickformat=',.0f', row=1, col=2)
fig.update_yaxes(tickprefix='$', tickformat=',.0f', row=2, col=1)
fig.write_html(f'{OUTPUT_DIR}/dashboard.html')
fig.show()
print('Dashboard saved → outputs/dashboard.html')

## 📋 7 — Business Summary Report

In [ ]:
print('=' * 52)
print('       BUSINESS PERFORMANCE SUMMARY')
print('=' * 52)
print(f"Total Revenue      : ${df['Revenue'].sum():>12,.2f}")
print(f"Total Profit       : ${df['Profit'].sum():>12,.2f}")
print(f"Avg Profit Margin  : {df['Profit_Margin_%'].mean():>11.1f}%")
print(f"Total Units Sold   : {df['Units_Sold'].sum():>12,}")
print(f"Total Orders       : {df['Order_ID'].count():>12,}")
print('-' * 52)
print(f"Best Category      : {df.groupby('Category')['Revenue'].sum().idxmax()}")
print(f"Best Product       : {df.groupby('Product')['Revenue'].sum().idxmax()}")
print(f"Best Region        : {df.groupby('Region')['Revenue'].sum().idxmax()}")
print(f"Top Sales Rep      : {df.groupby('Sales_Rep')['Revenue'].sum().idxmax()}")
print(f"Forecast Model     : {best_model_name}")
print('=' * 52)
print('\n✅ Analysis Complete!')